### **Simple ReAct Agent from Scratch**

In [18]:
import os, openai, re, httpx
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from pprint import pprint

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

class Agent:
    def __init__ (self, system = ''):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append(SystemMessage(content = self.system))
    
    def __call__ (self, message):
        self.messages.append(HumanMessage(content = message))
        response = self.execute()
        self.messages.append(AIMessage(content = response))
        return response
    
    def execute (self):
        response = ChatOpenAI(model = 'gpt-3.5-turbo', temperature = 0).invoke(self.messages)
        return response.content
    
def calculate(what):
    return eval(what)
    
def average_dog_weight(name):
    if name in 'Scottish Terrier': 
        return 'Scottish Terriers average 20 lbs'
    elif name in 'Border Collie':
        return 'a Border Collies average weight is 37 lbs'
    elif name in 'Toy Poodle':
        return 'a toy poodles average weight is 7 lbs'
    else:
        return 'An average dog weights 50 lbs'

known_actions = {
    'calculate': calculate,
    'average_dog_weight': average_dog_weight
}


In [4]:
prompt = '''
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
'''.strip()


In [6]:
AIbot = Agent(prompt)
response = AIbot('How much does a toy poodle weigh?')

print (response)

Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight action.
Action: average_dog_weight: Toy Poodle
PAUSE


In [10]:
response = average_dog_weight('Toy Poodle')
print(response)

a toy poodles average weight is 7 lbs


In [ ]:
next_prompt = 'Observation: {}'.format(response)
AIbot(next_prompt)

AIbot.messages

[SystemMessage(content='You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs', additional_kwargs={}, response_metadata={}),
 HumanMessag

In [46]:
question = '''I have 2 dogs, a border collie and a scottish terrier. What is their combined weight'''

AIbot = Agent(prompt)
response = AIbot(question)

print(response)

Thought: I can find the average weight of a Border Collie and a Scottish Terrier using the average_dog_weight action, then calculate their combined weight.

Action: average_dog_weight: Border Collie
PAUSE


In [47]:
next_prompt = 'Observation: {}'.format(average_dog_weight('Border Collie'))
response = AIbot(next_prompt)
print (response)

next_prompt = 'Observation: {}'.format(average_dog_weight('Scottish Terrier'))
response = AIbot(next_prompt)
print (response)

next_prompt = 'Observation: {}'.format(calculate('37 + 20'))
response = AIbot(next_prompt)
print (response)

Action: average_dog_weight: Scottish Terrier
PAUSE
Action: calculate: 37 + 20
PAUSE
Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs


#### **Add loop**

In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

def query(question, max_turns = 5):
    i = 0
    AIbot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        response = AIbot(next_prompt)
        print(response)

        actions = [
            action_re.match(a) 
            for a in response.split('\n') 
            if action_re.match(a)
        ]

        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception('Unknown action: {}: {}'.format(action, action_input))
            print(' -- running {} {}'.format(action, action_input))

            observation = known_actions[action](action_input)
            print('Observation:', observation)
            
            next_prompt = 'Observation: {}'.format(observation)
        else:
            print ('Total loops:', i)
            return

In [ ]:
query(question)

Thought: I can find the average weight of a Border Collie and a Scottish Terrier using the average_dog_weight action, and then calculate their combined weight.

Action: average_dog_weight: Border Collie
PAUSE
1   1 actions found
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Action: average_dog_weight: Scottish Terrier
PAUSE
2   1 actions found
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Action: calculate: 37 + 20
PAUSE
3   1 actions found
 -- running calculate 37 + 20
Observation: 57
Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs
4   0 actions found
Looping times: 4
